# Qwen3.5-9B LoRA+ Colab 원클릭 실행

1. Colab 메뉴에서 **런타임 → 런타임 유형 변경 → A100 GPU**를 선택합니다.
2. 이전에 실행한 vLLM이 있으면 **런타임 → 세션 다시 시작**을 먼저 누릅니다.
3. **런타임 → 모두 실행**을 한 번 누릅니다.

설치, 데이터 검증, vLLM 실행, rationale 생성, 3-shot 선정, LoRA+ 학습, validation 평가, 최종 재학습, 병합 및 선택적 Hugging Face 업로드를 순서대로 실행합니다. rationale와 체크포인트는 Google Drive에 저장되며, 다시 실행하면 완료된 단계를 건너뜁니다. 현재 BF16 구현은 A100 40GB 이상이 필요합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')
repo = Path('/content/Writing-assessment-ability')
if repo.is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/aaa0342/Writing-assessment-ability.git',
        str(repo),
    ], check=True)
os.chdir(repo)
ARTIFACTS_DIR = '/content/drive/MyDrive/writing-assessment-artifacts'
Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)
print('프로젝트:', Path.cwd())
print('영구 산출물:', ARTIFACTS_DIR)

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-U', 'uv'], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--system', '-U',
    'vllm', '--torch-backend=auto',
    '--extra-index-url', 'https://wheels.vllm.ai/nightly',
    '-r', 'requirements.txt', '-e', '.',
], check=True)
print('패키지 설치 완료')

In [ ]:
from google.colab import userdata

for secret_name in ('HF_TOKEN', 'HF_REPO_ID'):
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None
    if secret_value:
        os.environ[secret_name] = secret_value

LORAPLUS_RATIO = 16
EPOCHS = 3
print('LoRA+ ratio:', LORAPLUS_RATIO, 'epochs:', EPOCHS)
print('HF 업로드:', bool(os.environ.get('HF_TOKEN') and os.environ.get('HF_REPO_ID')))

In [ ]:
import sys

subprocess.run([
    sys.executable, '-m', 'essay_scorer', 'colab-run',
    '--train', '글쓰기채점능력평가2026_train.jsonl',
    '--validation', '글쓰기채점능력평가2026_validation.jsonl',
    '--artifacts', ARTIFACTS_DIR,
    '--ratio', str(LORAPLUS_RATIO),
    '--epochs', str(EPOCHS),
], check=True)